## Resources

In [ ]:
import os
import re
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from keras.models import Model
from keras.layers import Input, Dense, Conv2D, MaxPooling2D, Flatten, Dropout, LSTM, Bidirectional, concatenate, Layer
from keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score, precision_score, recall_score, f1_score
from keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K


mat_data_path = "/content/drive/MyDrive/utd"


class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1), initializer="normal")
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1), initializer="zeros")
        super(AttentionLayer, self).build(input_shape)

    def call(self, x):
        et = K.squeeze(K.tanh(K.dot(x, self.W) + self.b), axis=-1)
        at = K.softmax(et)
        at = K.expand_dims(at, axis=-1)
        output = x * at
        return K.sum(output, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])


def extract_label_from_filename(filename):
    match = re.search(r'a(\d+)', filename)
    if match:
        return int(match.group(1))
    else:
        raise ValueError(f"Cannot extract action ID from filename: {filename}")


def preprocess_skeleton(sample, target_len=76):
    # sample shape: (20, 3, T)
    T = sample.shape[2]
    if T > target_len:
        # truncate extra frames
        sample = sample[:, :, :target_len]
    elif T < target_len:
        # pad with zeros
        pad_width = target_len - T
        sample = np.pad(sample, ((0,0), (0,0), (0,pad_width)), mode='constant')
    return sample


def load_data(directory, data_key='d_skel', target_len=76):
    X, y = [], []

    for fname in sorted(os.listdir(directory)):
        if fname.endswith(".mat"):
            fpath = os.path.join(directory, fname)
            data = sio.loadmat(fpath)
            if data_key in data:
                sample = np.squeeze(data[data_key])
                if sample.shape[0:2] == (20, 3):   # (20,3,T)
                    sample = preprocess_skeleton(sample, target_len)
                    flat = sample.reshape(20, -1)  # (20, 3*76=228)
                    X.append(flat)
                    label = extract_label_from_filename(fname)
                    y.append(label)
                else:
                    print(f"Skipping {fname}: unexpected shape {sample.shape}")
    return np.array(X), np.array(y)


raw_X, raw_y = load_data(mat_data_path)
print(f"Loaded {len(raw_X)} valid samples.")


time_steps = 20
feature_dim = 228  # 3×76
X_cnn = raw_X.reshape(-1, 1, time_steps, feature_dim).astype('float32')
X_lstm = raw_X.reshape(-1, time_steps, feature_dim).astype('float32')
y_encoded = to_categorical(raw_y - 1)
num_classes = y_encoded.shape[1]


X_cnn_train = X_cnn
X_lstm_train = X_lstm
y_train = y_encoded

X_cnn_test = X_cnn
X_lstm_test = X_lstm
y_test = y_encoded


conv_input_layer = Input(shape=(1, time_steps, feature_dim))
conv_layer = Conv2D(6, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu')(conv_input_layer)
pooling_layer = MaxPooling2D(pool_size=(1,2), strides=(1,2), padding='same')(conv_layer)
flatten_layer = Flatten()(pooling_layer)

lstm_input_layer = Input(shape=(time_steps, feature_dim))
lstm_layer = Bidirectional(LSTM(64, return_sequences=True))(lstm_input_layer)
attention_output = AttentionLayer()(lstm_layer)

merged = concatenate([flatten_layer, attention_output])
dense_layer = Dense(64, activation='relu')(merged)
dropout_layer = Dropout(0.3)(dense_layer)
output_layer = Dense(num_classes, activation='softmax')(dropout_layer)

model = Model(inputs=[conv_input_layer, lstm_input_layer], outputs=output_layer)
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()


early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
history = model.fit(
    [X_cnn_train, X_lstm_train],
    y_train,
    epochs=220,
    batch_size=2,
    callbacks=[early_stop],
    verbose=1
)

score = model.evaluate([X_cnn_test, X_lstm_test], y_test, verbose=0)
print("Test Accuracy from evaluate():", score[1])

y_pred = model.predict([X_cnn_test, X_lstm_test])
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)


accuracy = accuracy_score(y_true, y_pred_classes)
precision = precision_score(y_true, y_pred_classes, average='weighted', zero_division=0)
recall = recall_score(y_true, y_pred_classes, average='weighted', zero_division=0)
f1 = f1_score(y_true, y_pred_classes, average='weighted', zero_division=0)

print("\n" + "="*40)
print(f"✅ TOTAL TEST ACCURACY: {accuracy * 100:.2f}%")
print(f"✅ WEIGHTED PRECISION : {precision * 100:.2f}%")
print(f"✅ WEIGHTED RECALL    : {recall * 100:.2f}%")
print(f"✅ WEIGHTED F1-SCORE  : {f1 * 100:.2f}%")
print("="*40 + "\n")

# ✅ Confusion Matrix
cm = confusion_matrix(y_true, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()

# ✅ Precision, Recall, F1-score Report
print("\nClassification Report (per class):\n")
print(classification_report(y_true, y_pred_classes, digits=2, zero_division=0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')